# Lunar Landing Site Navigator: precompute trên Google Colab

Notebook này chạy phần **tính toán nặng** (heavy run) của backend `lunarnav` (NASA Space Apps 2026,
"Lunar Landing Site Navigator"): vị trí Mặt Trời (Sun) và Trái Đất (Earth) so với **đường chân trời địa hình**
(terrain horizon), từ đó suy ra tiềm năng điện mặt trời (solar power) và cửa sổ liên lạc trực tiếp với Trái Đất
(direct-to-Earth, DTE). Frontend/UI làm sau, không có ở đây.

| Bước | Nội dung |
|---|---|
| 0 | `PARAMS`: chọn sites, khoảng thời gian, bước, providers, **`LIGHT_MODE`** |
| 1-3 | Lấy code (git clone hoặc zip bundle), `pip install -e`, mount Google Drive, đặt `LUNARNAV_DATA` / `LUNARNAV_OUTPUT` lên Drive |
| 4 | Tải Moon Trek LOLA DEM (Resources tab, 1-2 GB mỗi file) có progress + kiểm tra kích thước; bỏ qua nếu đã có |
| 5 | Chạy test suite (`pytest -m "not slow"`) |
| 6 | Precompute **A** (primary): JPL Horizons + Moon Trek DEM + raymarch |
| 7 | Precompute **B** (cross-check): NAIF SPICE + NASA GSFC PGDA DEM + bruteforce |
| 8 | `check_output` cho cả hai thư mục kết quả |
| 9 | So sánh A với B (ephemeris từng mẫu, % nắng / % Earth từng site) |
| 10 | Validation: `lunarnav.validate.run_all` (A/B/C + negative controls + Apollo 11 + lunar day) và `validate_terrain` |
| 11 | Moon Trek vs PGDA (độ cao, horizon mask, % chiếu sáng) |
| 12 | Bảng PASS/FAIL tổng hợp |
| 13 | Biểu đồ kiểm tra nhanh (sanity plots): sky diagram + timeline 1 tháng |
| 14 | Nén kết quả (zip) lên Drive |

**Cách dùng:** chạy lần đầu với `LIGHT_MODE = True` (vài phút, 2 sites, 3 ngày, chỉ PGDA, không tải Trek) để kiểm tra
môi trường. Sau đó đặt `LIGHT_MODE = False` và *Runtime > Run all* cho lần chạy đầy đủ (2026-2030, hàng giờ, toàn bộ
catalog). Nếu Colab ngắt kết nối, chạy lại: dữ liệu và kết quả nằm trên Drive, `--resume` bỏ qua các site đã xong.

Quy ước (conventions, xem `docs/OUTPUT_SPEC.md`): thời gian UTC; vĩ độ planetocentric, kinh độ **East** trong khung
MOON_ME; độ cao so với mặt cầu 1737.4 km; azimuth theo chiều kim đồng hồ từ Bắc (như Horizons); elevation airless apparent.

## 0. PARAMS

In [ ]:
# ======================= PARAMS: chỉnh ở đây (edit here) =======================
LIGHT_MODE = True   # True: smoke test vài phút. False: full run (nhiều giờ, tải ~4.6 GB Moon Trek DEM)

# --- Lấy code trên Colab (code source) ---
REPO_URL = ""        # git URL của repo; để trống -> dùng zip từ scripts/make_colab_bundle.py
REPO_BRANCH = ""     # branch/tag tuỳ chọn
BUNDLE_ZIP = ""      # đường dẫn zip; trống -> /content/lunarnav_colab_bundle.zip, rồi DRIVE_ROOT/lunarnav_colab_bundle.zip, rồi hộp upload
DRIVE_ROOT = "/content/drive/MyDrive/lunarnav"   # data/, output/, exports/ nằm ở đây (bền qua các phiên)
DEM_LOCAL_COPY = True   # Colab: chép DEM từ Drive sang ổ local /content để đọc ngẫu nhiên nhanh (bản gốc vẫn ở Drive)

# --- Phạm vi (scope) ---
SITES = "all"                      # catalog: south pole + reference + Apollo + far side (23 sites); hoặc "group:apollo,apollo11,..."
START_UTC = "2026-01-01T00:00:00"  # inclusive
END_UTC = "2031-01-01T00:00:00"    # exclusive -> 2026..2030
STEP_MIN = 60                      # hourly
OBSERVER_HEIGHT_M = 2.0            # mast; validate_terrain: >= 1-2 m (0 m observer tự che bóng trên DEM)
MIN_EARTH_CLEARANCE_DEG = 0.0
RADIUS_KM = 150.0                  # bán kính tìm horizon
AZ_STEP_DEG = 0.5

# --- Phương pháp (methods) ---
PRIMARY = dict(provider="horizons", dem="trek", horizon_method="raymarch")      # Method A + Moon Trek (Resources tab)
CROSSCHECK = dict(provider="spice", dem="pgda", horizon_method="bruteforce")    # Method B + PGDA (độc lập)
DEM_FALLBACK = "flat"   # sites ngoài vùng DEM cực (Apollo, far side): horizon mặt cầu nhẵn, ghi trong dem_source
TREK_LABELS = [         # Moon Trek LOLA polar DEMs mà get_dem('trek') dùng (5 m tới 60 km + 30 m; 100 m cho 45S..75S)
    "LRO_LOLA_DEM_SPole875_5mp_v04_EQ",    # 1.88 GB
    "LRO_LOLA_DEM_SPole75_30mp_v04_EQ",    # 1.03 GB
    "LRO_LOLA_DEM_SPole45_100mp_v04_EQ",   # 1.67 GB
]

# --- Kiểm tra (checks) ---
RUN_TESTS = True
TEST_MARKERS = "not slow"           # gồm cả network tests (cache trên Drive)
RUN_VALIDATION = True               # lunarnav.validate.run_all (ephemeris A/B/C)
RUN_TERRAIN_VALIDATION = True       # validate_terrain.run (~25 min cache lạnh, ~6 min cache nóng)
TREK_VS_PGDA_SITES = ["ref_connecting_ridge_c1", "ref_shackleton_rim", "a3_de_gerlache_rim_2", "a3_haworth"]
PLOT_SITE = "ref_connecting_ridge_c1"
PLOT_MONTH = "2026-03"              # timeline 1 tháng (YYYY-MM)

# --- Output ---
RUN_NAME = ""          # trống -> tự đặt: <mode>_<start>_<end>_<step>_<code hash>
RESUME = True          # --resume: bỏ qua site đã xong với cùng tham số + cùng code
ZIP_OUTPUT = True
FAIL_ON_ERROR = True   # ô cuối raise nếu có check bắt buộc FAIL

if LIGHT_MODE:  # ghi đè (overrides) cho smoke test: vài phút, dữ liệu nhỏ
    SITES = "ref_connecting_ridge_c1,apollo11"     # 1 site cực có địa hình + 1 site flat-fallback
    START_UTC, END_UTC = "2026-01-07T00:00:00", "2026-01-10T00:00:00"   # 3 ngày, có 1 lần Earth lặn (earthset) ở C1
    PRIMARY = dict(provider="horizons", dem="pgda", horizon_method="raymarch")   # PGDA only, không tải Trek
    CROSSCHECK = dict(provider="spice", dem="pgda", horizon_method="bruteforce")
    RADIUS_KM = 60.0
    TREK_LABELS = []
    TEST_MARKERS = "not slow and not network"
    RUN_TERRAIN_VALIDATION = False   # thay bằng kiểm tra terrain nhẹ ở bước 11
    TREK_VS_PGDA_SITES = ["ref_connecting_ridge_c1"]
    PLOT_MONTH = START_UTC[:7]
print("LIGHT_MODE =", LIGHT_MODE, "| sites:", SITES, "|", START_UTC, "->", END_UTC, f"step {STEP_MIN} min")
print("primary:", PRIMARY, "| cross-check:", CROSSCHECK)

## 1. Môi trường (environment) và Google Drive

In [ ]:
import os, sys, json, time, shutil, hashlib, zipfile, threading, subprocess, importlib, re, math
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
T_NOTEBOOK0 = time.time()
print("Colab:", IN_COLAB, "| Python", sys.version.split()[0], "|", sys.platform)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
    print("Drive root:", DRIVE_ROOT, "| free on Drive: %.1f GB" % (shutil.disk_usage(DRIVE_ROOT).free / 1e9))
else:
    print("Không chạy trên Colab: bỏ qua mount Drive (Drive-mount skipped).")

## 2. Lấy code (git clone hoặc zip bundle) và `pip install -e`

Trên Colab: nếu `REPO_URL` có giá trị thì `git clone`; nếu không thì dùng zip tạo bởi
`python scripts/make_colab_bundle.py` (chỉ code, không có data/output/.venv). Ngoài Colab: dùng repo chứa notebook
này (không cài gì).

In [ ]:
def _find_repo_root(start: Path):
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").is_file() and (p / "lunarnav" / "__init__.py").is_file():
            return p
    return None


def _safe_extract(zpath: Path, dest: Path) -> None:
    """Extract a bundle zip, refusing absolute paths and '..' entries."""
    with zipfile.ZipFile(zpath) as zf:
        for n in zf.namelist():
            if n.startswith(("/", "\\")) or ".." in Path(n).parts:
                raise ValueError(f"unsafe path in {zpath}: {n}")
        zf.extractall(dest)


if IN_COLAB:
    REPO_DIR = Path("/content/lunar-navigator")
    if REPO_URL:
        if (REPO_DIR / ".git").is_dir():
            subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
        else:
            if REPO_DIR.exists():
                shutil.rmtree(REPO_DIR)
            cmd = ["git", "clone", "--depth", "1"] + (["-b", REPO_BRANCH] if REPO_BRANCH else []) + [REPO_URL, str(REPO_DIR)]
            subprocess.run(cmd, check=True)
    else:
        cands = [Path(BUNDLE_ZIP)] if BUNDLE_ZIP else [Path("/content/lunarnav_colab_bundle.zip"),
                                                      Path(DRIVE_ROOT) / "lunarnav_colab_bundle.zip"]
        zpath = next((c for c in cands if c.is_file()), None)
        if zpath is None:
            from google.colab import files
            print("Không thấy bundle zip ở", [str(c) for c in cands], "-> hãy upload lunarnav_colab_bundle.zip")
            up = files.upload()
            if not up:
                raise FileNotFoundError("no bundle uploaded and REPO_URL is empty")
            zpath = Path("/content") / next(iter(up))
        if REPO_DIR.exists():
            shutil.rmtree(REPO_DIR)  # always start from the uploaded code
        _safe_extract(zpath, Path("/content"))
        if not (REPO_DIR / "lunarnav" / "__init__.py").is_file():
            raise FileNotFoundError(f"{zpath} does not contain lunar-navigator/lunarnav (made by make_colab_bundle.py?)")
        print("bundle:", zpath, (REPO_DIR / "BUNDLE_INFO.json").read_text())
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[dev,notebook]"],
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode != 0:
        raise RuntimeError("pip install -e failed (see above)")
else:
    REPO_DIR = Path(os.environ["LUNARNAV_REPO"]) if os.environ.get("LUNARNAV_REPO") else _find_repo_root(Path.cwd())
    if REPO_DIR is None:
        raise FileNotFoundError("run this notebook from inside the lunar-navigator checkout or set LUNARNAV_REPO")
    print("Không chạy trên Colab: bỏ qua clone/pip (pip cells skipped). Repo:", REPO_DIR)

# Import the code from REPO_DIR in this kernel (an editable install made after kernel start is
# only picked up by new processes; subprocesses below also get PYTHONPATH=REPO_DIR).
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
importlib.invalidate_caches()
import lunarnav
if Path(lunarnav.__file__).resolve().parent != (REPO_DIR / "lunarnav").resolve():
    raise RuntimeError(f"imported lunarnav from {lunarnav.__file__}, expected {REPO_DIR / 'lunarnav'}: restart the runtime")
print("lunarnav", lunarnav.__version__, "from", Path(lunarnav.__file__).parent)

## 3. Thư mục dữ liệu và kết quả (`LUNARNAV_DATA`, `LUNARNAV_OUTPUT`)

Trên Colab mọi thứ bền vững nằm trên Drive: `DRIVE_ROOT/data` (cache HTTP của Horizons/PGDA, SPICE kernels, DEM),
`DRIVE_ROOT/output` (kết quả precompute), `DRIVE_ROOT/exports` (zip). Với `DEM_LOCAL_COPY=True`,
`LUNARNAV_DATA=/content/lunarnav_data` là thư mục local trong đó `cache/` và `kernels/` là symlink sang Drive, còn `dem/`
là bản sao local của DEM trên Drive (đọc cửa sổ ngẫu nhiên từ Drive FUSE chậm). Ngoài Colab: dùng biến môi trường
đã đặt, hoặc `data/` và `output/` của repo.

In [ ]:
DRIVE_DEM_DIR = None   # persistent DEM copies on Drive when DEM_LOCAL_COPY (else dem/ is on Drive already)
if IN_COLAB:
    drive_data = Path(DRIVE_ROOT) / "data"
    for sub in ("cache", "kernels", "dem"):
        (drive_data / sub).mkdir(parents=True, exist_ok=True)
    if DEM_LOCAL_COPY:
        local_data = Path("/content/lunarnav_data")
        local_data.mkdir(parents=True, exist_ok=True)
        for sub in ("cache", "kernels"):
            link = local_data / sub
            if not link.is_symlink():
                if link.exists():
                    shutil.rmtree(link)
                link.symlink_to(drive_data / sub, target_is_directory=True)
        (local_data / "dem").mkdir(exist_ok=True)
        DRIVE_DEM_DIR = drive_data / "dem"
        os.environ["LUNARNAV_DATA"] = str(local_data)
    else:
        os.environ["LUNARNAV_DATA"] = str(drive_data)
    os.environ["LUNARNAV_OUTPUT"] = str(Path(DRIVE_ROOT) / "output")
    EXPORT_DIR = Path(DRIVE_ROOT) / "exports"
else:
    os.environ.setdefault("LUNARNAV_DATA", str(REPO_DIR / "data"))
    os.environ.setdefault("LUNARNAV_OUTPUT", str(REPO_DIR / "output"))
    EXPORT_DIR = Path(os.environ["LUNARNAV_OUTPUT"]) / "exports"

from lunarnav import config
from lunarnav.precompute import git_commit

DATA_DIR, OUTPUT_ROOT = config.data_root(), config.output_dir()
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


def code_sha256(repo: Path) -> str:
    """SHA-256 over the package sources (lunarnav/**/*.py + catalog JSON): identifies the code."""
    h = hashlib.sha256()
    files = sorted(list((repo / "lunarnav").rglob("*.py")) + list((repo / "lunarnav" / "catalog").glob("*.json")))
    for f in files:
        if "__pycache__" in f.parts:
            continue
        h.update(f.relative_to(repo).as_posix().encode() + b"\0" + f.read_bytes() + b"\0")
    return h.hexdigest()


CODE_SHA = code_sha256(REPO_DIR)
GIT_COMMIT = git_commit()
if not RUN_NAME:
    # The code identity is part of the name: --resume never mixes outputs of different code even
    # without git (the precompute fingerprint only sees __version__ + git commit).
    RUN_NAME = (f"{'light' if LIGHT_MODE else 'full'}_{START_UTC[:10]}_{END_UTC[:10]}_{int(STEP_MIN)}min_"
                f"{(GIT_COMMIT or CODE_SHA)[:8]}")
OUT_A = OUTPUT_ROOT / f"{RUN_NAME}_A_{PRIMARY['provider']}_{PRIMARY['dem']}_{PRIMARY['horizon_method']}"
OUT_B = OUTPUT_ROOT / f"{RUN_NAME}_B_{CROSSCHECK['provider']}_{CROSSCHECK['dem']}_{CROSSCHECK['horizon_method']}"
VAL_DIR = OUTPUT_ROOT / f"{RUN_NAME}_validation"
VAL_DIR.mkdir(parents=True, exist_ok=True)
print("LUNARNAV_DATA  =", os.environ["LUNARNAV_DATA"])
print("LUNARNAV_OUTPUT=", os.environ["LUNARNAV_OUTPUT"])
print("code sha256", CODE_SHA[:16], "| git commit", GIT_COMMIT)
print("run:", RUN_NAME, "\n  A ->", OUT_A, "\n  B ->", OUT_B, "\n  validation ->", VAL_DIR)

### Công cụ (helpers): chạy CLI, ghi kết quả kiểm tra

Mỗi kiểm tra được ghi vào `CHECKS` với kết quả `PASS` / `FAIL` (bắt buộc), `KNOWN` (sai khác đã biết và đã giải thích
trong docs, không tính là lỗi), `REVIEW` (tiêu chí mềm, cần người xem), `INFO`, `SKIP`.

In [ ]:
CHECKS = []


def record(section, check, value, limit, result, note=""):
    """Add one row to the pass/fail table (result: PASS FAIL KNOWN REVIEW INFO SKIP)."""
    assert result in ("PASS", "FAIL", "KNOWN", "REVIEW", "INFO", "SKIP"), result
    CHECKS.append(dict(section=section, check=check, value=str(value), limit=str(limit), result=result, note=note))
    return result


def show_checks(rows=None, title="PASS/FAIL"):
    rows = CHECKS if rows is None else rows
    w = [max(len(r[k]) for r in rows + [dict(section="section", check="check", value="value", limit="limit", result="result")])
         for k in ("section", "check", "value", "limit", "result")] if rows else [7] * 5
    w = [min(x, m) for x, m in zip(w, (24, 58, 44, 30, 6))]
    line = lambda r: "  ".join(str(r[k])[:wk].ljust(wk) for k, wk in zip(("section", "check", "value", "limit", "result"), w))
    print(f"== {title}: {len(rows)} checks ==")
    print(line(dict(section="section", check="check", value="value", limit="limit", result="result")))
    print("  ".join("-" * x for x in w))
    for r in rows:
        print(line(r) + (f"  ({r['note']})" if r["note"] else ""))
    counts = count_results(rows)
    print("totals:", counts)
    return counts


def count_results(rows=None):
    rows = CHECKS if rows is None else rows
    return {k: sum(r["result"] == k for r in rows) for k in ("PASS", "FAIL", "KNOWN", "REVIEW", "INFO", "SKIP")}


def run_cli(args, title=None):
    """Run `python -m lunarnav.cli <args>` in a subprocess with live output; return (rc, seconds)."""
    args = [str(a) for a in args]
    print("$ lunarnav " + " ".join(args), flush=True)
    env = dict(os.environ, PYTHONPATH=str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", ""),
               PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
    t0 = time.time()
    p = subprocess.Popen([sys.executable, "-m", "lunarnav.cli", *args], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace",
                         env=env, cwd=str(REPO_DIR))
    for ln in p.stdout:
        print(ln, end="", flush=True)
    rc = p.wait()
    dt = time.time() - t0
    print(f"-> exit code {rc} after {dt:.0f} s", flush=True)
    return rc, dt

## 4. Tải Moon Trek LOLA DEM (Resources tab)

Mỗi file 1-2 GB, chỉ tải trên Colab (`lunarnav.terrain.trek.download_product` từ chối file > 100 MB ở nơi khác).
Kiểm tra: kích thước phải đúng `fileSize` của catalogue Moon Trek, SHA-256 ghi vào `<file>.tif.json`, rồi mở file và kiểm
tra CRS (mặt cầu 1737.4 km), kiểu dữ liệu, kích thước lưới. Đã có (trên local hoặc Drive) thì bỏ qua. Moon Trek không hỗ trợ
`Range`, nên tải bị ngắt sẽ tải lại từ đầu.

In [ ]:
def _copy_with_progress(src: Path, dst: Path, size: int) -> None:
    tmp = dst.with_name(dst.name + ".copying")
    done, t0, nxt = 0, time.time(), 0.1
    with open(src, "rb") as fi, open(tmp, "wb") as fo:
        while True:
            b = fi.read(64 << 20)
            if not b:
                break
            fo.write(b)
            done += len(b)
            if done >= nxt * size:
                print(f"    copy {dst.name}: {100 * done / size:5.1f} % ({done / 1e6 / max(time.time() - t0, 1e-3):.0f} MB/s)", flush=True)
                nxt += 0.1
    if tmp.stat().st_size != size:
        raise RuntimeError(f"copy of {src} has {tmp.stat().st_size} bytes, expected {size}")
    os.replace(tmp, dst)


def ensure_trek_dem(label: str) -> Path:
    """Moon Trek product in config.dem_dir(): present -> keep; on Drive -> copy; else download (+ copy to Drive)."""
    from lunarnav.terrain import trek

    prod = trek.TREK_PRODUCTS[label]
    size = prod.file_size
    local_dir = config.dem_dir()
    local = local_dir / f"{label}.tif"
    ok = lambda p: p.is_file() and p.stat().st_size == size and Path(str(p) + ".json").is_file()
    if ok(local):
        how = "present"
    elif DRIVE_DEM_DIR is not None and ok(DRIVE_DEM_DIR / f"{label}.tif"):
        how = "copied from Drive"
        print(f"  {label}: copying {size / 1e9:.2f} GB from Drive to local disk", flush=True)
        _copy_with_progress(DRIVE_DEM_DIR / f"{label}.tif", local, size)
        shutil.copyfile(DRIVE_DEM_DIR / f"{label}.tif.json", Path(str(local) + ".json"))
    else:
        how = "downloaded"
        print(f"  {label}: downloading {size / 1e9:.2f} GB from Moon Trek (no resume support)", flush=True)
        part = local_dir / f"{label}.tif.part"
        stop = threading.Event()

        def _watch():
            t0 = time.time()
            while not stop.wait(20):
                n = part.stat().st_size if part.exists() else 0
                el = max(time.time() - t0, 1e-3)
                print(f"    {label}: {n / 1e6:9.1f} / {size / 1e6:.1f} MB ({100 * n / size:5.1f} %), {n / 1e6 / el:5.1f} MB/s", flush=True)

        th = threading.Thread(target=_watch, daemon=True)
        th.start()
        try:
            trek.download_product(label, dest_dir=local_dir)  # size check + sha256 sidecar + open/verify
        finally:
            stop.set()
            th.join()
        if DRIVE_DEM_DIR is not None:
            print(f"  {label}: saving a copy to Drive", flush=True)
            _copy_with_progress(local, DRIVE_DEM_DIR / f"{label}.tif", size)
            shutil.copyfile(Path(str(local) + ".json"), DRIVE_DEM_DIR / f"{label}.tif.json")
    got = local.stat().st_size
    meta = json.loads(Path(str(local) + ".json").read_text())
    trek.TrekDem(label=label, path=local).close()  # re-open: CRS sphere, dtype, grid size
    res = record("Moon Trek DEM", f"{label} size/sha256/open", f"{got} B, sha256 {meta.get('sha256', '')[:12]} ({how})",
                 f"== {size} B", "PASS" if got == size and meta.get("size") == size else "FAIL")
    print(f"  {label}: {res} ({how})", flush=True)
    return local


need_trek = "trek" in (PRIMARY["dem"], CROSSCHECK["dem"])
if not TREK_LABELS or not need_trek:
    print("Moon Trek download skipped (LIGHT_MODE or no 'trek' DEM selected).")
    record("Moon Trek DEM", "download", "skipped", "-", "SKIP", "LIGHT_MODE / not selected")
else:
    for lab in TREK_LABELS:
        try:
            ensure_trek_dem(lab)
        except Exception as exc:  # recorded, the precompute pre-flight then names the missing file
            record("Moon Trek DEM", f"{lab}", f"{type(exc).__name__}: {exc}", "file ok", "FAIL")
            print("ERROR", lab, exc)

## 5. Test suite (`pytest`)

In [ ]:
if RUN_TESTS:
    env = dict(os.environ, PYTHONPATH=str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", ""), PYTHONIOENCODING="utf-8")
    t0 = time.time()
    p = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-m", TEST_MARKERS, "-p", "no:cacheprovider", "tests"],
                       cwd=str(REPO_DIR), env=env, capture_output=True, text=True, encoding="utf-8", errors="replace")
    out = p.stdout + p.stderr
    tail = [ln for ln in out.splitlines() if ln.strip() and "Error in sys.excepthook" not in ln
            and not re.fullmatch(r"[.sxXFE]+\s*(\[\s*\d+%\])?", ln.strip())]  # drop progress-dot lines
    print("\n".join(tail[-25:]))
    summ = next((ln for ln in reversed(tail) if re.search(r"\d+ (passed|failed)", ln)), "no summary line")
    record("tests", f'pytest -m "{TEST_MARKERS}"', summ.strip("= ")[:60], "0 failed, exit 0",
           "PASS" if p.returncode == 0 else "FAIL", f"{time.time() - t0:.0f} s")
else:
    record("tests", "pytest", "skipped", "-", "SKIP")

## 6-7. Precompute A (primary) và B (cross-check)

A = JPL Horizons (Method A) + Moon Trek DEM + raymarch; B = NAIF SPICE (Method B) + PGDA DEM + bruteforce (thuật toán
horizon độc lập). Trước khi tính, CLI kiểm tra **mọi** site có DEM phục vụ được không (pre-flight) và dừng với danh sách
đầy đủ nếu thiếu file. Sites ngoài vùng DEM cực (Apollo, far side) nhận horizon mặt cầu nhẵn (`--dem-fallback flat`). Cuối
mỗi lần chạy CLI tự gọi `check_output`. Chạy lại ô này sau khi mất kết nối: `--resume` bỏ qua các site đã xong.

In [ ]:
def precompute_args(cfg, out):
    a = ["precompute", "--sites", SITES, "--start", START_UTC, "--end", END_UTC, "--step-min", STEP_MIN,
         "--provider", cfg["provider"], "--dem", cfg["dem"], "--horizon-method", cfg["horizon_method"],
         "--radius-km", RADIUS_KM, "--az-step", AZ_STEP_DEG, "--observer-height", OBSERVER_HEIGHT_M,
         "--min-clearance", MIN_EARTH_CLEARANCE_DEG, "--dem-fallback", DEM_FALLBACK, "--out", out]
    return a + (["--resume"] if RESUME else [])


RUNS = {}
for tag, cfg, out in (("A", PRIMARY, OUT_A), ("B", CROSSCHECK, OUT_B)):
    print(f"\n=========== precompute {tag}: {cfg} ===========")
    rc, dt = run_cli(precompute_args(cfg, out))
    RUNS[tag] = dict(rc=rc, seconds=dt, out=out, cfg=cfg)
    record("precompute", f"{tag}: {cfg['provider']} + {cfg['dem']} + {cfg['horizon_method']}",
           f"exit {rc}, {dt / 60:.1f} min", "exit 0 (incl. check_output)", "PASS" if rc == 0 else "FAIL")

## 8. `check_output` (mọi invariant của docs/OUTPUT_SPEC.md §4)

In [ ]:
from lunarnav.precompute import check_output, read_table, read_summary

for tag, out in (("A", OUT_A), ("B", OUT_B)):
    if not (out / "manifest.json").is_file():
        record("check_output", tag, "no manifest.json (precompute failed)", "0 violations", "FAIL")
        continue
    t0 = time.time()
    viol = check_output(out)
    for v in viol[:20]:
        print(f"  {tag} VIOLATION:", v)
    man = json.loads((out / "manifest.json").read_text())
    record("check_output", f"{tag}: {out.name}", f"{len(viol)} violations, {man['parameters']['n_samples']} samples x "
           f"{len(man['parameters']['site_ids'])} sites", "0 violations", "PASS" if not viol else "FAIL",
           f"{time.time() - t0:.0f} s")
_ = show_checks([r for r in CHECKS if r["section"] == "check_output"], "check_output")

## 9. So sánh A với B (cross-check trên chính kết quả precompute)

* **Ephemeris từng mẫu** (A = Horizons, B = SPICE): góc lệch lớn nhất giữa hai hướng Sun / Earth. Giới hạn = 0.02"
  (giới hạn A-B của `validate.py`; đo được ~0.002", do DE441 vs DE440) + 0.005" (hai lần làm tròn `%.6f` deg khi ghi file)
  + thị sai (parallax) do độ cao người quan sát A/B khác nhau (`|dh| / khoảng cách`, vì DEM A và B cho độ cao bề mặt
  khác nhau vài m).
* **Sites flat ở cả A và B:** horizon giống hệt nên cờ `sun_visible` / `earth_visible` phải trùng; chỉ cho phép khác ở mẫu
  cách ngưỡng < 0.001 deg (lớn hơn 100 lần sai khác A-B).
* **Sites có địa hình:** DEM khác (Trek vs PGDA) và thuật toán khác (raymarch vs bruteforce) nên chỉ là tiêu chí mềm:
  |Δ% nắng|, |Δ% Earth| ≤ 5 pp là `PASS`, lớn hơn là `REVIEW` (`validate_terrain` cho thấy dời site 100 m có thể đổi tới
  65 pp ở đỉnh núi).

In [ ]:
import numpy as np
from lunarnav.validate import angular_separation_deg

AB = []
if (OUT_A / "sites.json").is_file() and (OUT_B / "sites.json").is_file():
    SA = {r["id"]: r for r in json.loads((OUT_A / "sites.json").read_text())["sites"]}
    SB = {r["id"]: r for r in json.loads((OUT_B / "sites.json").read_text())["sites"]}
    sumA, sumB = read_summary(OUT_A / "summary.csv"), read_summary(OUT_B / "summary.csv")
    is_flat = lambda r: r["horizon_method"] == "flat" or str(r["dem_source"]).startswith("none")
    for sid in [s for s in SA if s in SB]:
        ra, rb = SA[sid], SB[sid]
        ea, eb = read_table(OUT_A / ra["files"]["ephemeris"]), read_table(OUT_B / rb["files"]["ephemeris"])
        if not np.array_equal(ea["time_utc"], eb["time_utc"]):
            record("A vs B", f"{sid} time grid", "different", "identical", "FAIL")
            continue
        dh_km = abs(ra["observer_height_above_sphere_m"] - rb["observer_height_above_sphere_m"]) / 1000.0
        row = dict(site=sid)
        for body in ("sun", "earth"):
            sep = angular_separation_deg(ea[f"{body}_az_deg"], ea[f"{body}_el_deg"], eb[f"{body}_az_deg"], eb[f"{body}_el_deg"])
            parallax_as = math.degrees(dh_km / float(np.min(ea[f"{body}_dist_km"]))) * 3600
            lim = 0.02 + 0.005 + parallax_as
            mx = float(np.max(sep)) * 3600
            row[f"{body}_sep_as"] = mx
            record("A vs B ephemeris", f"{sid} {body} max direction diff", f'{mx:.4f}"', f'{lim:.4f}" (dh {dh_km * 1000:.1f} m)',
                   "PASS" if mx <= lim else "FAIL")
        va, vb = read_table(OUT_A / ra["files"]["visibility"]), read_table(OUT_B / rb["files"]["visibility"])
        ia, ib = sumA[sid], sumB[sid]
        dpct = {k: float(ia[k]) - float(ib[k]) for k in ("pct_sun", "pct_earth", "pct_both")}
        row.update({f"d_{k}": v for k, v in dpct.items()})
        txt = ", ".join(f"{k[4:]} {float(ia[k]):.2f}/{float(ib[k]):.2f}" for k in dpct)
        if is_flat(ra) and is_flat(rb):
            margin_s = np.abs(va["sun_el_deg"] + va["sun_semidiam_deg"] - va["horizon_el_sun_deg"])
            margin_e = np.abs(va["earth_clearance_deg"] - MIN_EARTH_CLEARANCE_DEG)
            bad_s = (va["sun_visible"] != vb["sun_visible"]) & (margin_s >= 1e-3)
            bad_e = (va["earth_visible"] != vb["earth_visible"]) & (margin_e >= 1e-3)
            n_mis = int(np.sum(va["sun_visible"] != vb["sun_visible"]) + np.sum(va["earth_visible"] != vb["earth_visible"]))
            record("A vs B visibility", f"{sid} (flat both) flags A/B", f"{n_mis} flag diffs; % {txt}",
                   "diffs only < 0.001 deg from threshold", "PASS" if not (bad_s.any() or bad_e.any()) else "FAIL")
        elif not is_flat(ra) and not is_flat(rb):
            ha, hb = read_table(OUT_A / ra["files"]["horizon"]), read_table(OUT_B / rb["files"]["horizon"])
            p95 = float(np.percentile(np.abs(ha["el_deg"] - hb["el_deg"]), 95)) if ha["el_deg"].shape == hb["el_deg"].shape else float("nan")
            worst = max(abs(v) for v in dpct.values())
            record("A vs B terrain", f"{sid} % sun/earth/both A/B", f"{txt}; mask p95 {p95:.2f} deg",
                   "|d| <= 5 pp (soft)", "PASS" if worst <= 5.0 else "REVIEW",
                   f"{ra['dem_source'][:28]} vs {rb['dem_source'][:28]}")
        else:
            record("A vs B terrain", f"{sid} % sun/earth/both A/B", txt, "-", "INFO",
                   f"different terrain: {ra['dem_source'][:24]} vs {rb['dem_source'][:24]}")
        AB.append(row)
else:
    record("A vs B", "inputs", "A or B output missing", "-", "FAIL")
_ = show_checks([r for r in CHECKS if r["section"].startswith("A vs B")], "A vs B")

## 10. Validation: ephemeris A/B/C (`lunarnav.validate.run_all`) và terrain (`lunarnav.validate_terrain`)

`run_all` so sánh Horizons (A), SPICE (B) và mô hình giải tích ERFA/IAU (C) trên 6 sites x 2 tháng + 24 thời điểm ngẫu
nhiên, chạy 12 negative controls (sai khung, sai thang thời gian, sai aberration, sai dấu kinh độ...) để chứng minh giới hạn
đủ chặt, kiểm tra Sun elevation lúc Apollo 11 hạ cánh và độ dài ngày Mặt Trăng. Ở full mode, nó gọi thêm
`validate_terrain.run()` (raymarch vs bruteforce trên địa hình thật, chiếu sáng của ta so với bản đồ NASA PGDA, độ nhạy).
Báo cáo được lưu vào `VAL_DIR`. Giới hạn và lý do: `docs/VALIDATION.md`, `docs/VALIDATION_TERRAIN.md`.

In [ ]:
KNOWN_TERRAIN = {  # documented definition differences (docs/VALIDATION_TERRAIN.md section 2.3), not code errors
    ("surface0", "lit"): "0 m observer on the DEM shadows itself; NASA map ignores the nearest pixels",
    ("pixel60", "lit"): "0 m observer, near field < 60 m ignored: still darker than the NASA map",
}


def ephemeris_rows(eph):
    for s in eph["matrix"]["summary"]:
        thr = s["threshold"]
        record("ephemeris A/B/C", f"{s['pair']} {s['body']} max angle ({s['n_samples']} samples)",
               f'{s["sep"]["max"] * 3600:.4f}" / range {s["dist_km"]["max"] * 1000:.1f} m',
               f'{thr["sep_deg"] * 3600:g}"', "PASS" if s["passed"] else "FAIL")
    for v in eph.get("diagnostics", {}).get("variants", []):
        res = ("PASS" if v["passed"] else "FAIL") if v["must_detect"] else "INFO"
        record("negative control", v["description"][:58], f'Sun {v["sun_sep_max_arcsec"]:.3f}" Earth {v["earth_sep_max_arcsec"]:.3f}"',
               "must be caught" if v["must_detect"] else "info", res)
    if "apollo11" in eph:
        a = eph["apollo11"]
        vals = ", ".join(f"{k} {x:.3f}" for k, x in a["sun_el_deg"].items())
        record("historical fact", f"Apollo 11 landing Sun elevation {a['time_utc']}", vals,
               f"{a['documented_deg']} +- {a['tolerance_deg']} deg", "PASS" if a["passed"] else "FAIL")
    if "lunar_day" in eph:
        d = eph["lunar_day"]
        record("lunar day", f"daylight / sunset-to-sunset at {d['site']}",
               "; ".join(f"{x['daylight_true_h']:.1f} h" for x in d["days"]), "Kepler solar-day formula",
               "PASS" if d["passed"] else "FAIL")


def terrain_rows(t):
    """Rows from a validate_terrain.run() report (criteria from docs/VALIDATION_TERRAIN.md)."""
    if t.get("status") in ("error", "absent"):
        record("terrain", "validate_terrain", t.get("error") or t.get("note"), "ran", "FAIL" if t.get("status") == "error" else "SKIP")
        return
    ec = t.get("ephemeris_check")
    if ec:
        ok = ec["earth_el_deg"] < 1e-4 and ec["earth_dist_km"] < 0.5
        record("terrain", "Moon-centre vector ephemeris vs SpiceProvider", f"Earth el {ec['earth_el_deg']:.1e} deg, {ec['earth_dist_km']:.3f} km",
               "< 1e-4 deg, < 0.5 km", "PASS" if ok else "FAIL")
    for m in t.get("method_comparison", []):
        b = m["beyond_near_exclusion"]
        # bound slope*(res/2)/d ~ 0.075 deg (horizon.py report) -> 0.1 deg on p95; max 0.3 deg
        ok = b["p95_abs_deg"] < 0.1 and b["max_abs_deg"] < 0.3
        record("terrain", f"raymarch vs bruteforce {m['site_id']} (> {m['near_exclusion_m'] / 1000:g} km)",
               f"p95 {b['p95_abs_deg']:.3f}, max {b['max_abs_deg']:.3f} deg", "p95 < 0.1, max < 0.3 deg", "PASS" if ok else "FAIL")
    for s in t.get("sensitivity_mast", []):
        ok = s["monotone"] and s.get("illum_monotone", True)
        record("terrain", f"mast 2->10 m lowers horizon {s['site_id']}", f"drop {s['min_drop_deg']:.3f}..{s['max_drop_deg']:.3f} deg",
               "monotone in every bin", "PASS" if ok else "FAIL")
    for var, st in sorted(t.get("illumination", {}).get("stats", {}).items()):
        for kind in ("psr", "lit", "earth"):
            x = st.get(kind)
            if not x or "pass" not in x:
                continue
            if kind == "lit":
                val = f"bias {x['mean']:+.1f}, med|d| {x['median_abs']:.1f}, p90 {x['p90_abs']:.1f} pp (n={x['n']})"
                lim = "5 / 5 / 12 pp"
            elif kind == "psr":
                val = f"max {x.get('max_avg_illum_pct', float('nan')):.3f} %"
                lim = "< 1 %"
            else:
                val = f"mean|d| {x['mean_abs']:.2f} pp (n={x['n']})"
                lim = "< 10 pp"
            res = "PASS" if x["pass"] else ("KNOWN" if (var, kind) in KNOWN_TERRAIN else "FAIL")
            record("terrain vs NASA PGDA maps", f"{kind} pixels, observer '{var}'", val, lim, res,
                   KNOWN_TERRAIN.get((var, kind), "") if res == "KNOWN" else "")
    sc = t.get("step_check")
    if sc:
        d = abs(sc["avg_illum_pct_1h"] - sc["avg_illum_pct_6h"])
        record("terrain", f"1 h vs 6 h step ({sc['site_id']})", f"{d:.3f} pp", "< 0.5 pp", "PASS" if d < 0.5 else "FAIL")
    for lab, h in t.get("trek_header", {}).items():
        record("Moon Trek header", f"{lab} scale/CRS/dtype", f"PGDA/raw {h['ratio_pgda_over_raw_lsq']:.5f} (assumed {h['assumed_scale_to_m']})",
               "within 1 %, sphere CRS", "PASS" if h["pass"] else "FAIL")


VAL_REPORT = None
if RUN_VALIDATION:
    from lunarnav import validate
    t0 = time.time()
    try:
        VAL_REPORT = validate.run_all(include_horizons=True, offline=False, terrain=RUN_TERRAIN_VALIDATION,
                                      quick=LIGHT_MODE, log=lambda m: print(m, flush=True))
        (VAL_DIR / "validation_report.json").write_text(json.dumps(VAL_REPORT, indent=1, sort_keys=True))
        (VAL_DIR / "validation_report.md").write_text(validate.render_markdown(VAL_REPORT))
        ephemeris_rows(VAL_REPORT["ephemeris"])
        if "terrain" in VAL_REPORT:
            terrain_rows(VAL_REPORT["terrain"])
            rp = VAL_REPORT["terrain"].get("report_path")
            if rp and Path(rp).is_file():
                shutil.copyfile(rp, VAL_DIR / "validate_terrain_report.json")
        else:
            record("terrain", "validate_terrain.run", "not run here", "-", "SKIP",
                   "LIGHT_MODE: light terrain checks in step 11" if LIGHT_MODE else "RUN_TERRAIN_VALIDATION=False")
    except Exception as exc:
        record("validation", "lunarnav.validate.run_all", f"{type(exc).__name__}: {exc}"[:200], "runs", "FAIL")
        print("ERROR", repr(exc))
    print(f"validation took {time.time() - t0:.0f} s; reports in {VAL_DIR}")
else:
    record("validation", "run_all", "skipped", "-", "SKIP")
_ = show_checks([r for r in CHECKS if r["section"] in ("ephemeris A/B/C", "negative control", "historical fact", "lunar day",
                                                     "validation") or r["section"].startswith("terrain")], "validation")

## 11. Moon Trek vs PGDA (hai nguồn DEM LOLA độc lập)

`validate_terrain.trek_header_check` kiểm tra giả định về file Trek (scale nửa mét của Int16, CRS) chỉ từ vài MB đầu file.
`compare_dem_sources` so sánh độ cao, horizon mask (cùng điểm quan sát) và % chiếu sáng giữa Moon Trek và PGDA quanh từng
site. Full mode: các file Trek đã tải (`TrekDem`, lưới địa lý `_EQ`). LIGHT_MODE: Moon Trek ImageServer (lưới cực gốc 30 m,
đọc vài block). Tiêu chí (mềm, `REVIEW` nếu vượt): |median Δh| ≤ 2 m, mask ngoài 1 km p95 ≤ 0.5 deg, |Δ chiếu sáng| ≤ 3 pp.
Đã đo trên ImageServer: 0.14-0.27 m, 0.15-0.30 deg, 0.1-0.8 pp, nên ngưỡng có biên khoảng 2-4 lần. File `_EQ` 30 m thực ra có
độ phân giải theo phương bán kính 87 m, nên có thể lệch nhiều hơn: đó là lý do dùng `REVIEW`, không dùng `FAIL`.

In [ ]:
from lunarnav import validate_terrain as vt
from lunarnav.sites import get_site
from lunarnav.terrain.pgda import PgdaDem

TREK_PGDA = []
try:
    if not (VAL_REPORT and "trek_header" in VAL_REPORT.get("terrain", {})):
        for lab, h in vt.trek_header_check().items():
            record("Moon Trek header", f"{lab} scale/CRS/dtype", f"PGDA/raw {h['ratio_pgda_over_raw_lsq']:.5f} (assumed {h['assumed_scale_to_m']})",
                   "within 1 %, sphere CRS", "PASS" if h["pass"] else "FAIL")
    from lunarnav.terrain import trek
    have30 = (config.dem_dir() / "LRO_LOLA_DEM_SPole75_30mp_v04_EQ.tif").is_file()
    if have30 and not LIGHT_MODE:
        trek_src, radius_m = trek.TrekDem("LRO_LOLA_DEM_SPole75_30mp_v04_EQ", patch_resolution_m=30), 20_000.0
    else:
        trek_src, radius_m = trek.TrekImageServerDem("LRO_LOLA_DEM_SPole75_30mp_v04"), 10_000.0
    pgda30 = PgdaDem("LDEM_75S_30MPP_ADJ")
    eph_days = 365.25 if LIGHT_MODE else max(30.0, (np.datetime64(END_UTC) - np.datetime64(START_UTC)) / np.timedelta64(1, "D"))
    eph_tv = vt.MoonCentreEphemeris.compute(vt.make_times(START_UTC, eph_days, 6.0))
    for sid in TREK_VS_PGDA_SITES:
        try:
            r = vt.compare_dem_sources(trek_src, pgda30, get_site(sid), radius_m, eph=eph_tv)
        except Exception as exc:
            record("Moon Trek vs PGDA", sid, f"{type(exc).__name__}: {exc}"[:120], "runs", "FAIL")
            continue
        r.pop("_masks", None)
        TREK_PGDA.append(r)
        hd, mb = r["height_diff_m"], r.get("mask_diff_beyond_deg", {})
        dill = r["avg_illum_pct_a"] - r["avg_illum_pct_b"]
        ok = abs(hd["median"]) <= 2.0 and mb.get("p95_abs_deg", 0.0) <= 0.5 and abs(dill) <= 3.0
        record("Moon Trek vs PGDA", f"{sid} ({r['dem_a'].split(':')[0]}, {radius_m / 1000:g} km)",
               f"dh med {hd['median']:+.2f} std {hd['std']:.1f} m; mask>1km p95 {mb.get('p95_abs_deg', float('nan')):.2f}; "
               f"illum {r['avg_illum_pct_a']:.1f}/{r['avg_illum_pct_b']:.1f}",
               "2 m / 0.5 deg / 3 pp (soft)", "PASS" if ok else "REVIEW")
    (VAL_DIR / "trek_vs_pgda.json").write_text(json.dumps(TREK_PGDA, indent=1, sort_keys=True, default=str))
except Exception as exc:
    record("Moon Trek vs PGDA", "comparison", f"{type(exc).__name__}: {exc}"[:200], "runs", "FAIL")
    print("ERROR", repr(exc))
_ = show_checks([r for r in CHECKS if r["section"].startswith("Moon Trek")], "Moon Trek vs PGDA")

## 12. Bảng PASS/FAIL tổng hợp

In [ ]:
COUNTS = show_checks(title="ALL CHECKS")
with open(VAL_DIR / "checks.csv", "w", encoding="utf-8", newline="") as f:
    import csv
    w = csv.DictWriter(f, fieldnames=["section", "check", "value", "limit", "result", "note"])
    w.writeheader()
    w.writerows(CHECKS)
print("\nKNOWN = sai khác đã biết, có giải thích (docs/VALIDATION_TERRAIN.md); REVIEW = tiêu chí mềm, cần xem.")
print("Overall:", "PASS" if COUNTS["FAIL"] == 0 else f"FAIL ({COUNTS['FAIL']} required checks failed)")

if (OUT_A / "summary.csv").is_file():
    print("\nRanking (A):")
    print(f"{'rank':>4} {'site':<26} {'%sun':>7} {'%earth':>7} {'%both':>7} {'maxdark_h':>9} {'maxgap_h':>9}")
    for sid, r in sorted(read_summary(OUT_A / "summary.csv").items(), key=lambda kv: int(kv[1]["rank"])):
        print(f"{r['rank']:>4} {sid:<26} {float(r['pct_sun']):7.2f} {float(r['pct_earth']):7.2f} {float(r['pct_both']):7.2f} "
              f"{float(r['longest_dark_h']):9.1f} {float(r['longest_comms_gap_h']):9.1f}")

## 13. Biểu đồ kiểm tra nhanh (sanity plots)

Chỉ để kiểm tra bằng mắt, UI làm sau. (1) Sky diagram: đường chân trời địa hình (vùng xám) cùng quỹ đạo Sun / Earth trên
bầu trời trong cả khoảng thời gian. Ở cực nam, Sun phải nằm sát chân trời (khoảng ±1.6°) và đi đủ 360° azimuth mỗi tháng;
Earth nằm trong khoảng ±7° quanh azimuth ≈ 360° − lon. (2) Timeline 1 tháng: phần đĩa Mặt Trời nhìn thấy (sun fraction)
và Earth visible (DTE).

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

PLOTS = []
if (OUT_A / "sites.json").is_file():
    recs = {r["id"]: r for r in json.loads((OUT_A / "sites.json").read_text())["sites"]}
    rec = recs.get(PLOT_SITE) or next(iter(recs.values()))
    hz = read_table(OUT_A / rec["files"]["horizon"])
    vis = read_table(OUT_A / rec["files"]["visibility"])
    t = vis["time_utc"]

    fig, ax = plt.subplots(figsize=(11, 4.2))
    az = np.r_[hz["az_deg"], 360.0]
    el = np.r_[hz["el_deg"], hz["el_deg"][0]]
    ys = np.r_[el, vis["sun_el_deg"], vis["earth_el_deg"]]
    lo, hi = max(-90.0, np.nanmin(ys) - 1.0), min(90.0, np.nanmax(ys) + 1.0)
    ax.fill_between(az, lo, el, color="0.55", alpha=0.6, label="terrain horizon", step=None)
    sv = vis["sun_visible"]
    ax.scatter(vis["sun_az_deg"][sv], vis["sun_el_deg"][sv], s=3, color="#e8a200", label="Sun (visible)")
    ax.scatter(vis["sun_az_deg"][~sv], vis["sun_el_deg"][~sv], s=3, color="#8a6d3b", alpha=0.35, label="Sun (hidden)")
    ev = vis["earth_visible"]
    ax.scatter(vis["earth_az_deg"][ev], vis["earth_el_deg"][ev], s=3, color="#1f77b4", label="Earth (visible)")
    ax.scatter(vis["earth_az_deg"][~ev], vis["earth_el_deg"][~ev], s=3, color="#7fa7c9", alpha=0.35, label="Earth (hidden)")
    ax.set_xlim(0, 360); ax.set_ylim(lo, hi)
    ax.set_xticks(range(0, 361, 45)); ax.set_xticklabels(["N", "NE", "E", "SE", "S", "SW", "W", "NW", "N"])
    ax.set_xlabel("azimuth (clockwise from north)"); ax.set_ylabel("elevation (deg)")
    ax.set_title(f"Sky diagram {rec['id']} ({rec['lat_deg']:.3f}, {rec['lon_deg']:.3f} E), {rec['dem_source'][:50]}, "
                 f"{str(t[0])[:10]}..{str(t[-1])[:10]}", fontsize=9)
    ax.legend(loc="upper right", fontsize=7, markerscale=3, ncol=3); ax.grid(alpha=0.3)
    fig.tight_layout(); p = VAL_DIR / f"sky_{rec['id'].replace(':', '_')}.png"; fig.savefig(p, dpi=110); PLOTS.append(p)
    plt.show()

    m = np.array([str(x).startswith(PLOT_MONTH) for x in t])
    if not m.any():
        m[:] = True
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 4.6), sharex=True)
    a1.plot(t[m], vis["sun_fraction"][m], color="#e8a200", lw=1)
    bm = vis["both_visible"][m]
    a1.fill_between(t[m], 0, 1, where=bm | np.r_[False, bm[:-1]], color="green", alpha=0.12, step="post",
                    label="Sun AND Earth")  # sample k covers [t_k, t_k+1)
    a1.set_ylabel("sun fraction"); a1.set_ylim(-0.05, 1.05); a1.legend(loc="upper right", fontsize=7)
    a2.step(t[m], vis["earth_visible"][m].astype(int), where="post", color="#1f77b4")
    a2.set_ylabel("Earth visible"); a2.set_yticks([0, 1]); a2.set_ylim(-0.1, 1.1)
    a1.set_title(f"{rec['id']}: {PLOT_MONTH}  (sun {100 * vis['sun_visible'][m].mean():.1f} %, "
                 f"earth {100 * vis['earth_visible'][m].mean():.1f} %, both {100 * vis['both_visible'][m].mean():.1f} %)", fontsize=9)
    fig.autofmt_xdate(); fig.tight_layout(); p = VAL_DIR / f"timeline_{rec['id'].replace(':', '_')}_{PLOT_MONTH}.png"
    fig.savefig(p, dpi=110); PLOTS.append(p)
    plt.show()
    # quick numeric sanity of what the plot shows (polar site physics, docs/OUTPUT_SPEC.md section 4)
    if abs(rec["lat_deg"]) >= 89.0:
        smax = float(np.max(np.abs(vis["sun_el_deg"])))
        record("sanity", f"{rec['id']} |Sun el| max", f"{smax:.2f} deg", f"<= {1.65 + 90 - abs(rec['lat_deg']):.2f} deg",
               "PASS" if smax <= 1.65 + 90 - abs(rec["lat_deg"]) else "FAIL")
else:
    print("no output A to plot")
print("plots:", [str(p) for p in PLOTS])

## 14. Nén kết quả (zip) lên Drive

In [ ]:
EXPORT_ZIP = None
if ZIP_OUTPUT:
    EXPORT_ZIP = EXPORT_DIR / f"{RUN_NAME}.zip"
    tmp_zip = Path("/content" if IN_COLAB else str(EXPORT_DIR)) / f".{RUN_NAME}.zip.tmp"
    n = 0
    with zipfile.ZipFile(tmp_zip, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for d in (OUT_A, OUT_B, VAL_DIR):
            if not d.is_dir():
                continue
            for f in sorted(d.rglob("*")):
                if f.is_file() and not f.name.endswith(".tmp"):
                    zf.write(f, f"{RUN_NAME}/{d.name}/{f.relative_to(d).as_posix()}")
                    n += 1
        zf.writestr(f"{RUN_NAME}/RUN_INFO.json", json.dumps(dict(
            run_name=RUN_NAME, light_mode=LIGHT_MODE, sites=SITES, start_utc=START_UTC, end_utc=END_UTC,
            step_min=STEP_MIN, primary=PRIMARY, crosscheck=CROSSCHECK, code_sha256=CODE_SHA, git_commit=GIT_COMMIT,
            lunarnav_version=lunarnav.__version__, checks=count_results()), indent=1, sort_keys=True))
    shutil.move(str(tmp_zip), str(EXPORT_ZIP))
    h = hashlib.sha256(EXPORT_ZIP.read_bytes()).hexdigest()
    print(f"{EXPORT_ZIP}: {n} files, {EXPORT_ZIP.stat().st_size / 1e6:.2f} MB, sha256 {h[:16]}")
print(f"notebook time {(time.time() - T_NOTEBOOK0) / 60:.1f} min")

## Kết luận (verdict)

In [ ]:
failed = [r for r in CHECKS if r["result"] == "FAIL"]
print(f"{count_results()}\n")
for r in failed:
    print("FAIL:", r["section"], "|", r["check"], "|", r["value"])
if FAIL_ON_ERROR and failed:
    raise AssertionError(f"{len(failed)} required check(s) failed - see the table above")
print("All required checks passed." if not failed else "Some checks failed (FAIL_ON_ERROR=False).")